# 04 — Análise descritiva: SCR/BACEN + Atlas de Desastres

Este notebook reproduz, de forma organizada, as análises descritivas dos notebooks antigos do TCC.

**Importante:** este notebook **não faz ETL**. Ele parte das bases produzidas por:

1. `src/01_consolidar_scr.py`
2. `src/02_preparar_atlas.py`
3. `src/03_integrar_bases.py`

Período analítico: **2013-01 a 2024-12**  
Unidade da base principal: **UF × mês**

## 1. Bibliotecas e caminhos

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker

try:
    import plotly.express as px
    PLOTLY_OK = True
except ImportError:
    PLOTLY_OK = False

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

def encontrar_raiz(inicio: Path) -> Path:
    inicio = inicio.resolve()
    for pasta in [inicio, *inicio.parents]:
        if (pasta / "data").exists() and (pasta / "src").exists():
            return pasta
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto.")

ROOT = encontrar_raiz(Path.cwd())
PROCESSED = ROOT / "data" / "processed"
INTERIM = ROOT / "data" / "interim"
FIGURES = ROOT / "outputs" / "figures"
TABLES = ROOT / "outputs" / "tables"

FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

print("Raiz do projeto:", ROOT)

## 2. Leitura das bases

In [ ]:
FINAL_PARQUET = PROCESSED / "df_tcc_2013_2024.parquet"
FINAL_CSV = PROCESSED / "df_tcc_2013_2024.csv"

ATLAS_EVENTOS_PARQUET = INTERIM / "atlas_eventos_2013_2024.parquet"
ATLAS_EVENTOS_CSV = INTERIM / "atlas_eventos_2013_2024.csv"

if FINAL_PARQUET.exists():
    try:
        df = pd.read_parquet(FINAL_PARQUET)
    except (ImportError, ModuleNotFoundError):
        df = pd.read_csv(FINAL_CSV, sep=";")
else:
    df = pd.read_csv(FINAL_CSV, sep=";")

if ATLAS_EVENTOS_PARQUET.exists():
    try:
        atlas = pd.read_parquet(ATLAS_EVENTOS_PARQUET)
    except (ImportError, ModuleNotFoundError):
        atlas = pd.read_csv(ATLAS_EVENTOS_CSV, sep=";")
else:
    atlas = pd.read_csv(ATLAS_EVENTOS_CSV, sep=";")

df["data_base"] = pd.to_datetime(df["data_base"])
atlas["Data_Evento"] = pd.to_datetime(atlas["Data_Evento"])

print("Base final:", df.shape)
print("Atlas eventos:", atlas.shape)
display(df.head())

## 3. Validação da base principal

In [ ]:
print("Período:", df["mes_ano"].min(), "a", df["mes_ano"].max())
print("Meses:", df["mes_ano"].nunique())
print("UFs:", df["uf"].nunique())
print("Linhas:", len(df))
print("Duplicidades UF + mês:", df.duplicated(["uf", "mes_ano"]).sum())

assert df["mes_ano"].nunique() == 144
assert df["uf"].nunique() == 27
assert len(df) == 144 * 27
assert df.duplicated(["uf", "mes_ano"]).sum() == 0

## 4. Duas medidas de inadimplência

- **`taxa_inadimplencia`**: razão monetária entre carteira inadimplente e carteira ativa. Esta é a medida principal do TCC.
- **`pct_registros_com_inadimplencia`**: proporção dos registros originais do SCR em que `carteira_inadimplencia > 0`. É mantida apenas para reproduzir a análise descritiva inicial.

In [ ]:
display(
    df[
        [
            "taxa_inadimplencia",
            "pct_registros_com_inadimplencia",
            "carteira_ativa_total",
            "carteira_inadimplencia_total",
        ]
    ].describe()
)

## 5. Proporção de registros com inadimplência por UF e ano

In [ ]:
df_registros_ano = (
    df.groupby(["uf", "ano"], as_index=False)
    .agg(
        qtd_total=("qtd_registros", "sum"),
        qtd_maus=("qtd_maus", "sum"),
    )
)

df_registros_ano["inadimplencia_pct_registros"] = (
    df_registros_ano["qtd_maus"]
    / df_registros_ano["qtd_total"]
    * 100
)

tabela = df_registros_ano.pivot(
    index="ano",
    columns="uf",
    values="inadimplencia_pct_registros",
)

ax = tabela.plot(figsize=(14, 7), linewidth=1.2)
ax.set_title("Proporção de Registros com Inadimplência por UF")
ax.set_xlabel("Ano")
ax.set_ylabel("Registros com inadimplência (%)")
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.legend(ncol=3, fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.savefig(FIGURES / "pct_registros_inadimplencia_uf_ano.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Taxa monetária de inadimplência — todos os estados

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))

for uf, grupo in df.groupby("uf"):
    grupo = grupo.sort_values("data_base")
    ax.plot(
        grupo["data_base"],
        grupo["taxa_inadimplencia"],
        linewidth=1.2,
        alpha=0.85,
        label=uf,
    )

ax.xaxis.set_major_formatter(mdates.DateFormatter("%b/%y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")

ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
ax.set_title("Taxa de Inadimplência por Estado — PF (2013–2024)")
ax.set_xlabel("Mês/Ano")
ax.set_ylabel("Taxa de Inadimplência (%)")
ax.legend(ncol=9, fontsize=7.5, loc="upper center", bbox_to_anchor=(0.5, -0.22), frameon=False)
ax.grid(axis="y", linestyle="--", alpha=0.4)

fig.tight_layout()
plt.savefig(FIGURES / "inadimplencia_todos_estados_2013_2024.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Taxa de inadimplência por região — ponderada pela carteira ativa

In [ ]:
df_regiao_mes = (
    df.groupby(["regiao", "data_base"], as_index=False)
    .agg(
        carteira_ativa_total=("carteira_ativa_total", "sum"),
        carteira_inadimplencia_total=("carteira_inadimplencia_total", "sum"),
    )
)

df_regiao_mes["taxa_inadimplencia"] = (
    df_regiao_mes["carteira_inadimplencia_total"]
    / df_regiao_mes["carteira_ativa_total"]
    * 100
)

fig, ax = plt.subplots(figsize=(13, 6))

for regiao, grupo in df_regiao_mes.groupby("regiao"):
    grupo = grupo.sort_values("data_base")
    ax.plot(
        grupo["data_base"],
        grupo["taxa_inadimplencia"],
        linewidth=2,
        label=regiao,
    )

ax.xaxis.set_major_formatter(mdates.DateFormatter("%b/%y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")

ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
ax.set_title("Taxa de Inadimplência por Região — PF (2013–2024)")
ax.set_xlabel("Mês/Ano")
ax.set_ylabel("Taxa de Inadimplência (%)")
ax.legend(frameon=False)
ax.grid(axis="y", linestyle="--", alpha=0.4)

fig.tight_layout()
plt.savefig(FIGURES / "inadimplencia_regioes_2013_2024.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Minas Gerais — pico e vale

In [ ]:
df_mg = df.loc[df["uf"].eq("MG")].sort_values("data_base")

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(df_mg["data_base"], df_mg["taxa_inadimplencia"], linewidth=2.2)

idx_max = df_mg["taxa_inadimplencia"].idxmax()
idx_min = df_mg["taxa_inadimplencia"].idxmin()

pico_x = df_mg.loc[idx_max, "data_base"]
pico_y = df_mg.loc[idx_max, "taxa_inadimplencia"]
vale_x = df_mg.loc[idx_min, "data_base"]
vale_y = df_mg.loc[idx_min, "taxa_inadimplencia"]

ax.annotate(
    f"Pico: {pico_y:.2f}%\n{pico_x.strftime('%b/%Y')}",
    xy=(pico_x, pico_y),
    xytext=(30, 10),
    textcoords="offset points",
    arrowprops=dict(arrowstyle="->"),
)

ax.annotate(
    f"Vale: {vale_y:.2f}%\n{vale_x.strftime('%b/%Y')}",
    xy=(vale_x, vale_y),
    xytext=(30, -20),
    textcoords="offset points",
    arrowprops=dict(arrowstyle="->"),
)

ax.xaxis.set_major_formatter(mdates.DateFormatter("%b/%y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")

ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f%%"))
ax.set_title("Taxa de Inadimplência — Minas Gerais (PF, 2013–2024)")
ax.set_xlabel("Mês/Ano")
ax.set_ylabel("Taxa de Inadimplência (%)")
ax.grid(axis="y", linestyle="--", alpha=0.4)

fig.tight_layout()
plt.savefig(FIGURES / "inadimplencia_MG_2013_2024.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Taxa monetária anual de inadimplência por UF

In [ ]:
df_inad_uf_ano = (
    df.groupby(["uf", "regiao", "ano"], as_index=False)
    .agg(
        carteira_ativa=("carteira_ativa_total", "sum"),
        carteira_inadimplencia=("carteira_inadimplencia_total", "sum"),
    )
)

df_inad_uf_ano["taxa_inadimplencia"] = (
    df_inad_uf_ano["carteira_inadimplencia"]
    / df_inad_uf_ano["carteira_ativa"]
    * 100
)

df_inad_uf_ano.to_csv(
    TABLES / "inadimplencia_uf_ano_2013_2024.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

display(df_inad_uf_ano.head(20))

## 10. Mapa de inadimplência por UF — opcional

In [ ]:
ANO_MAPA = 2024

if PLOTLY_OK:
    try:
        import requests

        df_mapa = df_inad_uf_ano.loc[
            df_inad_uf_ano["ano"].eq(ANO_MAPA),
            ["uf", "taxa_inadimplencia"],
        ].copy()

        url = (
            "https://raw.githubusercontent.com/codeforamerica/"
            "click_that_hood/master/public/data/brazil-states.geojson"
        )

        resposta = requests.get(url, timeout=30)
        resposta.raise_for_status()
        geojson = resposta.json()

        fig_mapa = px.choropleth(
            df_mapa,
            geojson=geojson,
            locations="uf",
            featureidkey="properties.sigla",
            color="taxa_inadimplencia",
            color_continuous_scale="Reds",
            title=f"Taxa de Inadimplência por Estado — {ANO_MAPA}",
            labels={"taxa_inadimplencia": "Inadimplência (%)"},
        )

        fig_mapa.update_geos(fitbounds="locations", visible=False)
        fig_mapa.show()

    except Exception as exc:
        print("Mapa não gerado:", exc)
else:
    print("Mapa não gerado porque Plotly não está instalado.")

## 11. Distribuição dos desastres por grupo

In [ ]:
contagem_grupos = (
    atlas["grupo_de_desastre"]
    .value_counts(dropna=False)
    .rename_axis("grupo_de_desastre")
    .reset_index(name="quantidade")
)

display(contagem_grupos)

contagem_grupos.to_csv(
    TABLES / "desastres_por_grupo_2013_2024.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

## 12. Distribuição dos desastres por tipologia

In [ ]:
contagem_tipologias = (
    atlas["descricao_tipologia"]
    .value_counts(dropna=False)
    .rename_axis("descricao_tipologia")
    .reset_index(name="quantidade")
)

display(contagem_tipologias)

contagem_tipologias.to_csv(
    TABLES / "desastres_por_tipologia_2013_2024.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

## 13. Desastres por grupo, UF e ano

In [ ]:
df_grupo_ano = (
    atlas.groupby(["uf", "ano", "grupo_de_desastre"], observed=True)
    .size()
    .reset_index(name="quantidade")
)

display(df_grupo_ano.head(30))

if PLOTLY_OK:
    fig = px.bar(
        df_grupo_ano,
        x="uf",
        y="quantidade",
        color="grupo_de_desastre",
        facet_col="ano",
        facet_col_wrap=3,
        barmode="stack",
        title="Quantidade de Desastres por Grupo — UF e Ano (2013–2024)",
        labels={
            "uf": "UF",
            "quantidade": "Ocorrências",
            "grupo_de_desastre": "Grupo",
        },
        height=700,
    )
    fig.update_xaxes(tickangle=90)
    fig.show()

## 14. Conferência da contagem de desastres na base integrada

In [ ]:
colunas_grupo = [c for c in df.columns if c.startswith("grupo_")]
colunas_tipo = [c for c in df.columns if c.startswith("tipo_")]

soma_grupos = df[colunas_grupo].sum().sort_values(ascending=False)
soma_tipos = df[colunas_tipo].sum().sort_values(ascending=False)

print("=== Grupos ===")
display(soma_grupos.to_frame("quantidade"))
print("Total grupos:", int(soma_grupos.sum()))

print("\n=== Tipologias ===")
display(soma_tipos.to_frame("quantidade"))
print("Total tipologias:", int(soma_tipos.sum()))

assert int(soma_grupos.sum()) == len(atlas)
assert int(soma_tipos.sum()) == len(atlas)

## 15. Desastres × inadimplência por UF — 2013–2024

In [ ]:
inad_uf = (
    df.groupby(["uf", "regiao"], as_index=False)
    .agg(media_inadimplencia=("taxa_inadimplencia", "mean"))
)

desastres_uf = (
    df.groupby("uf", as_index=False)
    .agg(total_desastres_periodo=("total_desastres", "sum"))
)

df_scatter = inad_uf.merge(
    desastres_uf,
    on="uf",
    how="left",
    validate="one_to_one",
)

display(df_scatter.sort_values("total_desastres_periodo", ascending=False))

if PLOTLY_OK:
    fig = px.scatter(
        df_scatter,
        x="total_desastres_periodo",
        y="media_inadimplencia",
        text="uf",
        color="regiao",
        size="total_desastres_periodo",
        size_max=50,
        title="Total de Desastres × Taxa Média de Inadimplência por UF (2013–2024)",
        labels={
            "total_desastres_periodo": "Total de desastres registrados",
            "media_inadimplencia": "Taxa média de inadimplência (%)",
            "regiao": "Região",
        },
        height=600,
    )
    fig.update_traces(textposition="top center")
    fig.show()

## 16. Correlação descritiva por UF

In [ ]:
correlacao_uf = df_scatter[
    ["total_desastres_periodo", "media_inadimplencia"]
].corr(method="pearson").iloc[0, 1]

print(
    "Correlação de Pearson entre total de desastres "
    f"e inadimplência média por UF: {correlacao_uf:.4f}"
)

## 17. Desastres × inadimplência por região e ano

In [ ]:
df_reg_ano = (
    df.groupby(["regiao", "ano"], as_index=False)
    .agg(
        carteira_ativa=("carteira_ativa_total", "sum"),
        carteira_inadimplencia=("carteira_inadimplencia_total", "sum"),
        total_desastres=("total_desastres", "sum"),
    )
)

df_reg_ano["taxa_inadimplencia"] = (
    df_reg_ano["carteira_inadimplencia"]
    / df_reg_ano["carteira_ativa"]
    * 100
)

display(df_reg_ano.sort_values(["ano", "regiao"]))

if PLOTLY_OK:
    fig = px.scatter(
        df_reg_ano,
        x="total_desastres",
        y="taxa_inadimplencia",
        color="regiao",
        symbol=df_reg_ano["ano"].astype(str),
        size="total_desastres",
        size_max=40,
        text="regiao",
        title="Desastres × Inadimplência por Região e Ano (2013–2024)",
        labels={
            "total_desastres": "Total de desastres no ano",
            "taxa_inadimplencia": "Taxa de inadimplência (%)",
            "regiao": "Região",
        },
        height=650,
    )
    fig.update_traces(textposition="top center")
    fig.show()

## 18. Tabela região × ano

In [ ]:
tabela_regiao_ano = (
    df_reg_ano[
        [
            "regiao",
            "ano",
            "carteira_ativa",
            "carteira_inadimplencia",
            "taxa_inadimplencia",
            "total_desastres",
        ]
    ]
    .sort_values(["ano", "regiao"])
    .reset_index(drop=True)
)

tabela_regiao_ano.to_csv(
    TABLES / "tabela_regiao_ano_2013_2024.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

display(tabela_regiao_ano)

## 19. Tabela UF × ano

In [ ]:
df_uf_ano = (
    df.groupby(["uf", "regiao", "ano"], as_index=False)
    .agg(
        carteira_ativa=("carteira_ativa_total", "sum"),
        carteira_inadimplencia=("carteira_inadimplencia_total", "sum"),
        total_desastres=("total_desastres", "sum"),
    )
)

df_uf_ano["taxa_inadimplencia"] = (
    df_uf_ano["carteira_inadimplencia"]
    / df_uf_ano["carteira_ativa"]
    * 100
)

df_uf_ano.to_csv(
    TABLES / "tabela_uf_ano_2013_2024.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

display(df_uf_ano.sort_values(["ano", "uf"]).head(40))

## 20. Ranking de desastres por UF

In [ ]:
ranking_desastres_uf = (
    df.groupby(["uf", "regiao"], as_index=False)
    .agg(total_desastres=("total_desastres", "sum"))
    .sort_values("total_desastres", ascending=False)
    .reset_index(drop=True)
)

display(ranking_desastres_uf)

ranking_desastres_uf.to_csv(
    TABLES / "ranking_desastres_uf_2013_2024.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

## 21. Ranking de inadimplência média mensal por UF

In [ ]:
ranking_inad_uf = (
    df.groupby(["uf", "regiao"], as_index=False)
    .agg(taxa_media_inadimplencia=("taxa_inadimplencia", "mean"))
    .sort_values("taxa_media_inadimplencia", ascending=False)
    .reset_index(drop=True)
)

display(ranking_inad_uf)

ranking_inad_uf.to_csv(
    TABLES / "ranking_inadimplencia_uf_2013_2024.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

# Resultado

Ao final deste notebook:

- a base principal continua em `data/processed/df_tcc_2013_2024.*`;
- gráficos ficam em `outputs/figures/`;
- tabelas ficam em `outputs/tables/`;
- nenhuma transformação estrutural das bases é feita no notebook.

As próximas etapas metodológicas devem ficar em notebooks próprios, por exemplo:

- `05_estacionariedade.ipynb`
- `06_causalidade_granger.ipynb`
- `07_defasagens_distribuidas.ipynb`
- `08_sarimax.ipynb`